# Update state-level data center loadsite inputs (all states, 3 EPRI scenarios)

This notebook rebuilds the state-level data center loadsite inputs for all states, using EPRI Powering Intelligence data.

**Data source**: State-level Data Center Power Data from EPRI Power Intelligence 2026 — https://powering-intelligence.epri.com/dashboard/

**Output files** (in `inputs/load/`):
- `loadsite_st_epri_low_extended_to_2032.csv`
- `loadsite_st_epri_medium_extended_to_2032.csv`
- `loadsite_st_epri_high_extended_to_2032.csv`

**Steps of the analysis**:
- From the raw EPRI export, keep only years **2026-2030** (the `Historical` scenario and years before 2026 are dropped).
- Keep only **state, year, peak load**, converted from GW to **MW**.
- Drop `AK`, `HI`, and the `US` national-total row — ReEDS only models the contiguous US at the state level here.
- Extrapolate **2031 and 2032** by applying the last observed year-over-year growth rate (**2029 → 2030**), held constant for both extrapolated years (i.e. compounded: 2031 = 2030 × rate, 2032 = 2031 × rate). States with zero peak load in 2029 are held at their 2030 value.
- Generate one CSV per EPRI scenario (Low / Medium / High) in the `loadsitereg,t,MW` long format expected by ReEDS.

## 1. Setup

In [1]:
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "runreeds.py").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

EPRI_CSV_PATH = REPO_ROOT / "CEPM/preprocessing/datacenter_load_forecast/EPRI Powering Intelligence - All States and Total (2026-08-19).csv"
OUTPUT_DIR = REPO_ROOT / "inputs/load"

EPRI_YEARS = [2026, 2027, 2028, 2029, 2030]
EXTRAPOLATED_YEARS = [2031, 2032]
EXCLUDE_STATES = ["AK", "HI", "US"]

# EPRI scenario name -> output file suffix
SCENARIOS = {"Low": "low", "Medium": "medium", "High": "high"}

## 2. Read and clean EPRI data

In [2]:
epri = pd.read_csv(EPRI_CSV_PATH)

epri["Scenario"] = epri["Scenario"].astype(str).str.strip()
epri["State"] = epri["State"].astype(str).str.strip()
epri["Year"] = pd.to_numeric(epri["Year"], errors="raise").astype(int)
epri["Peak Load (GW)"] = pd.to_numeric(epri["Peak Load (GW)"], errors="raise")

epri.head()

,Scenario,State,Year,Nominal Capacity (GW),Peak Load (GW),Annual Energy (TWh)
0,Historical,AK,2021,0.000,0.000,0.000
1,Historical,AK,2022,0.000,0.000,0.000
2,Historical,AK,2023,0.000,0.000,0.000
3,Historical,AK,2024,0.000,0.000,0.000
4,Historical,AL,2021,0.017,0.017,0.147


Only `State`, `Year`, and `Peak Load` are kept. Only the years 2026-2030 are kept.

In [3]:
epri_clean = epri[
    epri["Scenario"].isin(SCENARIOS.keys())
    & epri["Year"].isin(EPRI_YEARS)
    & ~epri["State"].isin(EXCLUDE_STATES)
][["Scenario", "State", "Year", "Peak Load (GW)"]].copy()

# GW -> MW
epri_clean["MW"] = epri_clean["Peak Load (GW)"] * 1000

print(f"{epri_clean['State'].nunique()} states, {epri_clean['Year'].nunique()} years, {epri_clean['Scenario'].nunique()} scenarios")
epri_clean.head()

48 states, 5 years, 3 scenarios


,Scenario,State,Year,Peak Load (GW),MW
213,Low,AL,2026,0.267,267.0
214,Low,AL,2027,0.311,311.0
215,Low,AL,2028,0.358,358.0
216,Low,AL,2029,0.380,380.0
217,Low,AL,2030,0.401,401.0


## 3. Extrapolate 2031 and 2032

For each state and scenario, compute the 2029 → 2030 growth rate and hold it constant going forward:

$$\text{rate} = \frac{\text{MW}_{2030}}{\text{MW}_{2029}} \qquad \text{MW}_{2031} = \text{MW}_{2030} \times \text{rate} \qquad \text{MW}_{2032} = \text{MW}_{2031} \times \text{rate}$$

If a state has zero capacity in 2029, it is held flat at its 2030 value instead.

In [4]:
def extrapolate_scenario(scenario_df):
    """scenario_df: rows for a single scenario, all states, years 2026-2030."""
    pivot = scenario_df.pivot(index="State", columns="Year", values="MW")

    rate = (pivot[2030] / pivot[2029]).where(pivot[2029] != 0, other=1.0)

    pivot[2031] = pivot[2030] * rate
    pivot[2032] = pivot[2031] * rate

    long = pivot.reset_index().melt(id_vars="State", var_name="Year", value_name="MW")
    return long.sort_values(["State", "Year"]).reset_index(drop=True)


extrapolated = {
    scenario: extrapolate_scenario(epri_clean[epri_clean["Scenario"] == scenario])
    for scenario in SCENARIOS
}

extrapolated["Low"].head(10)

,State,Year,MW
0,AL,2026,267.000000
1,AL,2027,311.000000
2,AL,2028,358.000000
3,AL,2029,380.000000
4,AL,2030,401.000000
5,AL,2031,423.160526
6,AL,2032,446.545713
7,AR,2026,4.000000
8,AR,2027,6.000000
9,AR,2028,7.000000


## 4. Write the 3 loadsite CSVs

In [ ]:
HEADER_COMMENT = (
    "# Other region hierarchy levels (st; transreg; transgrp; interconnect; etc) can be used\n"
    "# in the first column; they should match the level in the file title: loadsite_{level}_{name}.csv.\n"
    "# Values for multiple states/transgrps/etc. in a given year should be added in long format.\n"
    "*loadsitereg,t,MW\n"
)

for scenario, suffix in SCENARIOS.items():
    loadsite = extrapolated[scenario].rename(columns={"State": "loadsitereg", "Year": "t"})
    loadsite = loadsite[["loadsitereg", "t", "MW"]]

    out_path = OUTPUT_DIR / f"loadsite_st_epri_{suffix}_extended_to_2032.csv"
    csv_text = HEADER_COMMENT + loadsite.to_csv(index=False, header=False)
    out_path.write_text(csv_text)

    print(f"Wrote {out_path} ({len(loadsite)} rows)")

Wrote /Users/monmac/Desktop/CEPM/ReEDS-CEPM/inputs/load/loadsite_st_epri_low_extended_to_2032.csv (336 rows)
Wrote /Users/monmac/Desktop/CEPM/ReEDS-CEPM/inputs/load/loadsite_st_epri_medium_extended_to_2032.csv (336 rows)
Wrote /Users/monmac/Desktop/CEPM/ReEDS-CEPM/inputs/load/loadsite_st_epri_high_extended_to_2032.csv (336 rows)
